In [9]:
from pathlib import Path

DATA_PREPROCESSED = Path("datasets/processed")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
# SPLITS = ["train", "val", "test"]
SPLITS = ["train"] 

# class mapping
CLASS_NAMES = {
    0: "no_animal",
    1: "red_deer",
    2: "roe_deer",
    3: "chamois",
    4: "human",
    5: "alpine_ibex",
    6: "fallow_deer",
    7: "unknown",
    8: "dog",
    9: "bird",
    10: "wild_boar",
    11: "hybrid_pig"
}

In [10]:
from pathlib import Path
from collections import Counter
import numpy as np
import csv

images_total = 0
images_with_animals = 0
images_without_animals = 0

animals_per_image = []

class_counter = Counter()

In [11]:
for split in SPLITS:
    label_dir = DATA_PREPROCESSED / LABELS_DIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        images_total += 1

        with open(file, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # Case: empty or only "0" → no animal image
        if len(lines) == 0 or (len(lines) == 1 and lines[0].split()[0] == "0"):
            images_without_animals += 1
            animals_per_image.append(0)
            continue

        # image contains animals
        images_with_animals += 1
        animals_per_image.append(len(lines))

        for line in lines:
            cls = int(line.split()[0])

            # if cls == 0:
            #     continue

            class_counter[cls] += 1

total_animals = sum(class_counter.values())
avg_animals_per_image = total_animals / images_total if images_total else 0

animals_array = np.array(animals_per_image)

In [12]:
print(f"Total images: {images_total}")
print(f"Images with animals: {images_with_animals}")
print(f"Images without animals: {images_without_animals}")
print(f"Percentage with animals: {images_with_animals / images_total * 100:.2f}%")

print("\n--- Animal stats ---")
print(f"Total animals: {total_animals}")
print(f"Average animals per image: {avg_animals_per_image:.2f}")
print(f"Max animals in one image: {animals_array.max()}")

print(f"95th percentile animals/image: {np.percentile(animals_array, 95):.2f}")
print(f"99th percentile animals/image: {np.percentile(animals_array, 99):.2f}")

print("\n--- Per class distribution ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    name = CLASS_NAMES[cls_id]
    count = class_counter.get(cls_id, 0)

    print(f"{cls_id:2d} {name:15s}: {count}")


Total images: 6710
Images with animals: 6449
Images without animals: 261
Percentage with animals: 96.11%

--- Animal stats ---
Total animals: 15060
Average animals per image: 2.24
Max animals in one image: 26
95th percentile animals/image: 6.00
99th percentile animals/image: 18.00

--- Per class distribution ---
 0 no_animal      : 1284
 1 red_deer       : 9221
 2 roe_deer       : 608
 3 chamois        : 716
 4 human          : 1413
 5 alpine_ibex    : 653
 6 fallow_deer    : 1165
 7 unknown        : 0
 8 dog            : 0
 9 bird           : 0
10 wild_boar      : 0
11 hybrid_pig     : 0


In [30]:
print("\n--- Class imbalance (percentage, FULL) ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    print(f"{CLASS_NAMES[cls_id]:15s}: {pct:.2f}%")


--- Class imbalance (percentage, FULL) ---
no_animal      : 4.45%
red_deer       : 63.95%
roe_deer       : 4.22%
chamois        : 4.97%
human          : 9.80%
alpine_ibex    : 4.53%
fallow_deer    : 8.08%
unknown        : 0.00%
dog            : 0.00%
bird           : 0.00%
wild_boar      : 0.00%
hybrid_pig     : 0.00%


In [31]:
print("\n--- Rare classes (<5%) ---")

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    if pct < 5:
        print(f"{CLASS_NAMES[cls_id]:15s}: {count:5d} - {pct:.2f}%")


--- Rare classes (<5%) ---
no_animal      :  1284 - 4.45%
roe_deer       :  1216 - 4.22%
chamois        :  1432 - 4.97%
alpine_ibex    :  1306 - 4.53%
unknown        :     0 - 0.00%
dog            :     0 - 0.00%
bird           :     0 - 0.00%
wild_boar      :     0 - 0.00%
hybrid_pig     :     0 - 0.00%


In [32]:
with open("analysis/dataset_class_distribution_preprocessed.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "class_name", "count", "percentage"])

    for cls_id, count in sorted(class_counter.items()):
        pct = (count / total_animals) * 100 if total_animals else 0
        writer.writerow([cls_id, CLASS_NAMES.get(cls_id), count, pct])

print("\nCSV saved: dataset_class_distribution_preprocessed.csv")


CSV saved: dataset_class_distribution_preprocessed.csv
